# 3. Offset (Idle 시간 보정) — 알고리즘 교육 자료

CMP 장비가 한동안 쉬었다가(Idle) 다시 가동하면, 패드 표면 상태·슬러리 온도 등이
변해 **첫 웨이퍼들의 실제 RR 이 패드 마모 모델(RR 학습값)의 예측과 달라집니다.**
그대로 연마하면 첫 웨이퍼들의 두께가 목표에서 벗어나므로,
Idle 유형별로 **연마 시간을 몇 초 보정해야 하는지(OFFSET)** 를 학습합니다.

핵심 아이디어 — 목표 제거량(`delta`)을 깎는 데 걸리는 시간을 두 가지로 계산해 비교:

```
실측 필요 시간   = delta / RR      (이 웨이퍼의 실제 연마율 기준)
모델 예측 시간   = delta / RR_Pad  (RR 학습값 b1/b0 가 예측하는 연마율 기준)

OFFSET = 실측 필요 시간 − 모델 예측 시간   (초)
```

Idle 직후 RR 이 떨어져 있었다면 실측 시간이 더 걸리므로 OFFSET > 0 —
다음에 같은 Idle 상황이 오면 APC 가 그만큼 연마 시간을 **가산**합니다.

이 노트북은 실제 프로덕션 코드 `algorithm_new/Common/OFFSET.py` 의 함수들을
**그대로 호출**하면서 단계별로 확인합니다.

**사용 방법: 아래 [실행 설정] 셀에서 `FAMILY` 와 `OPER_DESC` 두 값만 원하는 공정으로
바꾸고 위에서부터 순서대로 실행하세요.** Offset 학습은 RR 학습값(교육 자료 2편)을
입력으로 사용하는데, [준비] 셀이 자동으로 선행 실행해 줍니다.

```
전체 흐름  (Common/Module.py 의 compute_offset 과 동일한 순서)
────────────────────────────────────────────────────────────────────
 Set-up 조회 (FB_Type=TIME 행만)  +  merge_df 조회
   │
   ├─ [준비]   Pre_Thk_VM → RR 학습 선행 실행 (운영 파이프라인 순서 재현)
   │
   ├─ [Step 1] load_rr_data()      RR 학습값(B1/B0)을 merge_df 에 결합
   │
   ├─ [Step 2] OFFSET 산식          RR 실측 / RR_Pad 예측 → OFFSET (clip −5~3)
   │
   ├─ [Step 3] Idle_/Layer_ 구간    장비×레시피×IDLE 그룹 5건 이상 → 평균 저장
   │
   └─ [Step 4] compute_lc_offset()  LC 계열(패드 교체·Truing 등) 별도 집계
────────────────────────────────────────────────────────────────────
```

## 0. 환경 설정

- `algorithm_new/` 를 import 경로에 추가하고, Jupyter(async 이벤트 루프) 위에서
  Django ORM 동기 쿼리를 허용하도록 `DJANGO_ALLOW_ASYNC_UNSAFE` 를 설정합니다.
- 그래프는 기본 matplotlib 스타일 대신, 여백/그리드/색상을 정돈한 커스텀 스타일
  (`style_ax`, `PALETTE`)을 한 번만 정의하고 이후 모든 그래프에 재사용합니다.

In [ ]:
import sys, os
from pathlib import Path

ALGO_DIR = Path('..') / 'algorithm_new'
sys.path.insert(0, str(ALGO_DIR))

# Jupyter 는 async 이벤트 루프 위에서 실행되므로 Django ORM 동기 쿼리 허용
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'

import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)  # findfont 경고 숨김

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ── 한글 폰트 자동 탐색 (Mac/Windows/Linux 어디서 열어도 한글이 깨지지 않도록) ──
_KOREAN_FONTS = ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']
_installed = {f.name for f in font_manager.fontManager.ttflist}
_font = next((f for f in _KOREAN_FONTS if f in _installed), None)
# 한글 폰트를 1순위로, DejaVu Sans를 2순위로 두면 Å 같은 특수기호는
# 한글 폰트에 없어도 자동으로 DejaVu Sans에서 대체 렌더링됨 (matplotlib 폰트 폴백)
plt.rcParams['font.family'] = [_font, 'DejaVu Sans'] if _font else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ── 깔끔한 트렌드 스타일 (기본 matplotlib 스타일 대신 사용) ──────────────
INK_PRIMARY   = '#0b0b0b'   # 제목/본문
INK_SECONDARY = '#52514e'   # 축 라벨
INK_MUTED     = '#898781'   # 눈금
GRID          = '#e1e0d9'   # 그리드선 (아주 옅게)
BASELINE      = '#c3c2b7'   # 축선
SURFACE       = '#fcfcfb'   # 차트 배경

# 카테고리 색상은 항상 같은 순서로 고정 사용 (장비/채널이 늘어나도 임의로 섞지 않음)
PALETTE = ['#2a78d6', '#1baf7a', '#eda100', '#008300',
           '#4a3aa7', '#e34948', '#e87ba4', '#eb6834']
RAW_COLOR = '#b7d3f6'   # 원본(raw) 산점 — 옅은 블루, 라인과 구분

plt.rcParams.update({
    'figure.facecolor': SURFACE,
    'axes.facecolor'  : SURFACE,
    'axes.edgecolor'  : BASELINE,
    'axes.labelcolor' : INK_SECONDARY,
    'text.color'      : INK_PRIMARY,
    'xtick.color'     : INK_MUTED,
    'ytick.color'     : INK_MUTED,
    'font.size'       : 10,
    'figure.dpi'      : 110,
})

def style_ax(ax, title=None, ylabel=None, xlabel=None, legend=True, legend_loc='best'):
    # 모든 그래프에 공통 적용하는 정돈된 스타일 (spine 정리 + 옅은 그리드 + 좌측 정렬 제목)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_color(BASELINE)
        ax.spines[spine].set_linewidth(0.8)
    ax.set_axisbelow(True)
    ax.grid(axis='y', color=GRID, linewidth=0.8)
    ax.tick_params(labelsize=8.5, length=0)
    if title:
        ax.set_title(title, fontsize=11, fontweight='bold', color=INK_PRIMARY, loc='left', pad=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=9)
    if legend and ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8, frameon=False, loc=legend_loc)
    return ax

print(f'환경 설정 완료 (한글 폰트: {_font or "시스템 기본값"})')

## ★ 실행 설정 — 여기만 수정하세요

| 변수 | 설명 |
|---|---|
| `FAMILY` | `'DRAM'` 또는 `'NAND'` |
| `OPER_DESC` | 공정 설명 (web Set-up 의 Category.oper_desc 와 동일하게) |
| `KEY_INDEX` | 같은 공정에 실행 키(Lot_Code_Oper_Code_Fab)가 여러 개일 때 선택. 아래 [1. Set-up 조회] 결과 표의 번호 |
| `APC_INDEX` | Offset 학습 단위(FB_Type=TIME 인 Recipe × APC_Para)가 여러 개일 때 선택. [Step 2] 결과 표의 번호 |

In [ ]:
FAMILY    = 'DRAM'         # ← 원하는 Family
OPER_DESC = 'M1 CU CMP'    # ← 원하는 공정명

KEY_INDEX = 0              # 실행 키가 여러 개일 때 선택 (기본: 첫 번째)
APC_INDEX = 0              # Offset 학습 단위가 여러 개일 때 선택 (기본: 첫 번째)

print(f'실행 대상: Family={FAMILY} | Oper_Desc={OPER_DESC}')

## 1. Set-up 조회 & 실행 키 선택

`Get_data.baseinfoGetData()` 가 web Set-up(Django DB)에서 해당 공정의 Set-up 정보를
읽어옵니다. 운영 코드와 동일하게 **`Lot_Code + Oper_Code + Fab`** 조합으로 실행 키를
만들고, 키가 여러 개면 `KEY_INDEX` 로 하나를 선택합니다.

**Offset 은 `FB_Type == 'TIME'` 행만 사용합니다** — OFFSET 이 "연마 **시간** 보정량"
이므로 시간으로 제어하는 파라미터에만 적용되기 때문입니다
(`Module.py compute_offset()` 첫 줄과 동일).

In [ ]:
from Common.Get_Data import Get_data
from Common.OFFSET import OFFSET_Get

# ── Set-up 정보 로드 (Django DB) ─────────────────────────────────────
mico_info_table = Get_data.baseinfoGetData(Family=FAMILY, oper_desc=OPER_DESC)
mico_info_table['Group_Name']   = mico_info_table['Group_Name'].fillna('not_group')
mico_info_table['for_key_list'] = (
    mico_info_table['Lot_Code'] + '_' + mico_info_table['Oper_Code'] + '_' + mico_info_table['Fab']
)

# ── 실행 키 목록 표시 ────────────────────────────────────────────────
key_list = mico_info_table['for_key_list'].unique()
key_summary = pd.DataFrame([{
    '실행 키'   : k,
    'Group'     : mico_info_table.loc[mico_info_table['for_key_list'] == k, 'Group_Name'].iloc[0],
    'Recipe 수' : mico_info_table.loc[mico_info_table['for_key_list'] == k, 'Recipe_ID'].nunique(),
    'Detail 행' : (mico_info_table['for_key_list'] == k).sum(),
} for k in key_list])
print(f'실행 키 {len(key_list)}개 (KEY_INDEX={KEY_INDEX} 선택됨)')
display(key_summary)

# ── KEY_INDEX 로 실행 키 선택 ────────────────────────────────────────
assert 0 <= KEY_INDEX < len(key_list), f'KEY_INDEX 는 0 ~ {len(key_list)-1} 범위로 지정하세요'
FOR_KEY       = key_list[KEY_INDEX]
mico_info_key = mico_info_table[mico_info_table['for_key_list'] == FOR_KEY].copy()

Fab       = mico_info_key['Fab'].unique()[0]
Lot_Code  = mico_info_key['Lot_Code'].unique()[0]
Oper_Code = mico_info_key['Oper_Code'].unique()[0]
Oper_Desc = mico_info_key['Oper_Desc'].unique()[0]
Family    = mico_info_key['Family'].unique()[0]

# pol_type: Category.pol_type (web DB set-up 값) — Module.py _run_pipeline 과 동일
pol_type_vals = mico_info_key['Pol_Type'].dropna().unique()
pol_type = int(pol_type_vals[0]) if len(pol_type_vals) > 0 else None

# ── Offset 은 FB_Type=TIME 행만 사용 ─────────────────────────────────
mico_time     = mico_info_key[mico_info_key['FB_Type'] == 'TIME'].copy()
APC_Para_List = mico_time['APC_Para'].unique()
Offset_Group  = mico_time['Offset_Group'].unique()[0]
assert not mico_time.empty, 'FB_Type=TIME 행이 없습니다 — Offset 학습 대상이 아닙니다.'

print(f'\n선택된 키: {FOR_KEY}  (pol_type={pol_type})')
print(f'Offset 대상 APC_Para: {list(APC_Para_List)}  |  Offset_Group={Offset_Group}')
mico_time[['Recipe_ID', 'APC_Para', 'Thk_Para', 'Target', 'Pre_Target', 'Offset_Group']]

## 2. CMP 실측 데이터(merge_df) 조회

`Get_data.MongoDB_GetData()` 로 해당 공정의 CMP 실측 데이터를 조회합니다.
(로컬 테스트 환경에서는 이 함수가 `merge_df_sample.csv` 를 대신 읽도록 되어 있고,
회사 서버에서는 실제 MongoDB 쿼리가 실행됩니다 — 노트북 코드는 동일합니다.)

Offset 학습의 핵심 입력은 **IDLE 라벨**입니다. Merge 단계(Merge_Hub)가 각 웨이퍼에
"직전에 장비가 어떤 상태였는가"를 붙여 둡니다:

| 라벨 | 의미 |
|---|---|
| (빈 값) | 정상 연속 가동 |
| `Idle_1~4` | Idle 시간 구간별 (숫자가 클수록 오래 쉼) |
| `Layer_1~4` | 다른 Layer 연마 직후 재개 구간별 |
| `LC_…` | 패드 교체(Life Change) 직후 |
| `LC_T_… / LC_TB_…` | 패드 교체 + Truing (OFFSET=0 고정 처리) |

In [ ]:
merge_df = Get_data.MongoDB_GetData(Family, Fab, Lot_Code, Oper_Desc)
assert merge_df is not None and not merge_df.empty, '조회된 데이터가 없습니다. Set-up/기간을 확인하세요.'

merge_df['IDLE'] = merge_df['IDLE'].fillna('')
merge_df = merge_df[merge_df['operation_id'] == Oper_Code].copy()
assert not merge_df.empty, f'operation_id == {Oper_Code} 데이터가 없습니다.'

print(f'merge_df : {merge_df.shape[0]:,}행  (기간 {merge_df["Date"].min()} ~ {merge_df["Date"].max()})')
print(f'CMP 장비 : {sorted(merge_df["eqp_id"].unique())}')
print(f'\nIDLE 라벨 분포:')
display(merge_df['IDLE'].replace('', '(정상)').value_counts().rename('건수').to_frame())

## 3. 준비 — Pre_Thk_VM → RR 학습 선행 실행

OFFSET 산식의 분모에 들어가는 `RR_Pad` 는 **RR 학습값(B1/B0, 교육 자료 2편)** 으로
계산합니다. 운영 환경에서는 파이프라인이 항상 `Pre_Thk_VM → RR → Offset` 순서로 돌기
때문에 Offset 학습 시점에 RR 학습값이 MongoDB 에 준비되어 있습니다.

이 노트북에서는 같은 순서를 재현하기 위해 앞 두 단계를 직접 실행해
mock 저장소에 적재합니다. (출력이 길지만 각 단계의 저장 로그입니다 —
회사 서버에서는 이미 저장된 값을 읽기만 하므로 이 셀이 필요 없습니다.)

In [ ]:
from Common.Module import Module_Get, _MONGO_URL, _MONGO_DB
from Common.MongoDB_Control import _STORE

# 셀 재실행 대비: mock 저장소 초기화 (회사 서버에서는 MongoDB 에 누적 저장되므로 불필요)
for k in list(_STORE):
    _STORE[k] = []

print('━━ ① Pre_Thk_VM 학습 (교육 자료 1편) ━━')
Module_Get.compute_pre_thk_vm(merge_df, mico_info_key, pol_type)

print('\n━━ ② Removal Rate 학습 (교육 자료 2편) ━━')
Module_Get.compute_removal_rate(merge_df.copy(), mico_info_key, pol_type)

rr_collection = 'MICO_Removal_Rate_' + Lot_Code + '_' + Oper_Desc + '_' + Fab
RR_Table_raw  = pd.DataFrame(_STORE.get(rr_collection, []))
assert not RR_Table_raw.empty, 'RR 학습값이 없습니다 — 2편 노트북으로 원인을 확인하세요.'

# ── [로컬 전용] RR 학습 Date 보정 ────────────────────────────────────
# load_rr_data 는 merge_asof(backward)로 "웨이퍼 시점 이전의" RR 학습값을 붙입니다.
# 운영에서는 학습값이 매일 적재돼 항상 과거 학습값이 존재하지만, 로컬 샘플은 과거
# 데이터라 방금(오늘) 학습한 값이 모든 웨이퍼보다 미래가 되어 매칭되지 않습니다.
# → 학습 Date 를 데이터 시작 이전으로 당겨 매칭되게 합니다. (회사 서버에서는 불필요)
_backdate = merge_df['Date'].min() - pd.Timedelta(days=1)
for _row in _STORE[rr_collection]:
    _row['Date'] = _backdate

print(f'\nRR 학습값 {len(RR_Table_raw)}건 적재 완료 (컬렉션: {rr_collection})')
display(RR_Table_raw[[c for c in ['APC_Para', 'EQ', 'Recipe_ID', 'Count', 'b1', 'b0',
                                  'b1_weighted', 'b0_weighted'] if c in RR_Table_raw.columns]].head(8))

## Step 1 — `load_rr_data()` : RR 학습값(B1/B0)을 merge_df 에 결합

```python
# OFFSET.py  load_rr_data()
RR_Table[['b1_new', 'b0_new']] = RR_Table.apply(_get_b_coef, axis=1)
#   _get_b_coef: b1_weighted 가 있으면 weighted 사용, 없으면 기본 b1/b0

RR_pivot = pivot_table(index=['Date', 'eqp_id', 'recipe_id'],
                       columns='APC_Para', values=['b1_new', 'b0_new'])
#   → 컬럼명: {APC_Para}_B1, {APC_Para}_B0

merge_df = pd.merge_asof(merge_df.sort_values('Date'), RR_pivot.sort_values('Date'),
                         on='Date', by=['eqp_id', 'recipe_id'])
#   각 웨이퍼에 "그 시점 기준 가장 최근" RR 학습값을 결합
```

- **weighted 우선**: 최근 경향이 반영된 계수가 있으면 그것이 "현재의 정상 RR" 에
  더 가깝기 때문입니다.
- **merge_asof**: RR 학습값은 매일 갱신되므로, 웨이퍼별로 그 당시 유효했던 계수를
  시점 기준으로 붙입니다.

In [ ]:
merge_df_off = OFFSET_Get.load_rr_data(merge_df.copy(), Fab, Lot_Code, Oper_Desc,
                                       APC_Para_List, _MONGO_URL, _MONGO_DB)

b_cols = sorted([c for c in merge_df_off.columns if c.endswith('_B1') or c.endswith('_B0')])
print(f'추가된 계수 컬럼: {b_cols}')
for c in b_cols:
    n_ok = merge_df_off[c].notna().sum()
    print(f'  {c}: {n_ok:,}/{len(merge_df_off):,}행 매칭 ({n_ok / len(merge_df_off) * 100:.1f}%)')
print('(매칭은 RR 학습값이 존재하는 장비×레시피에만 됩니다 — 미매칭 행은 Step 2 에서 평균으로 보간)')

display(merge_df_off[['Date', 'eqp_id', 'recipe_id'] + b_cols].head(5))

## Step 2 — OFFSET 산식: 실측 RR vs 모델 예측 RR

`compute_offset()` 내부의 핵심 계산입니다. `APC_INDEX` 로 학습 단위 하나를 골라
따라갑니다.

```python
# compute_offset() 내부
Pol_Time = APC_Para 연마 시간 합
RR       = (Pre_Target − Thk_Para) / Pol_Time            # 실측 연마율 (REV 는 부호 반대)
RR_Pad   = 소모량 × {APC}_B1 + {APC}_B0                  # 패드 모델이 예측하는 정상 연마율

delta    = Pre_Target − Target                            # 목표 제거량
OFFSET   = delta/RR − delta/RR_Pad                        # 실측 필요시간 − 예측 시간 (초)

OFFSET   = OFFSET.clip(-5, 3)                             # 과보정 방지 (초 단위 상하한)
OFFSET[IDLE 가 LC_T_/LC_TB_] = 0                          # Truing 후에는 보정 없음
```

**clip(−5, 3) 을 하는 이유**: OFFSET 은 다음 웨이퍼의 연마 시간에 그대로 더해지는
값이라, 이상 계측 1건이 큰 OFFSET 을 만들면 후속 웨이퍼가 연달아 불량이 됩니다.
경험적으로 정상 Idle 영향 범위인 −5초 ~ +3초로 제한합니다.

In [ ]:
# ── 학습 단위 선택 ───────────────────────────────────────────────────
search_key = mico_time.reset_index(drop=True)
print(f'Offset 학습 단위 {len(search_key)}개 (APC_INDEX={APC_INDEX} 선택됨)')
display(search_key[['Recipe_ID', 'APC_Para', 'Thk_Para', 'Target', 'Pre_Target']])

assert 0 <= APC_INDEX < len(search_key), f'APC_INDEX 는 0 ~ {len(search_key)-1} 범위로 지정하세요'
key = search_key.iloc[APC_INDEX]

APC_Para   = key['APC_Para']
Thk_Para   = key['Thk_Para']
Recipe_ID  = key['Recipe_ID']
Pre_Target = float(key['Pre_Target'])
Target     = float(key['Target'])

Pol_Para = Get_data.APCParaGet(APC_Para, pol_type)
Pad_Para = Get_data.PadParaGet(APC_Para)
print(f'\n선택: Recipe={Recipe_ID} | APC_Para={APC_Para} | Pad_Para={Pad_Para}')
print(f'Pre_Target={Pre_Target} | Target={Target} → 목표 제거량 delta={Pre_Target - Target:.0f} Å')

In [ ]:
# ── compute_offset() 내부 로직 재현 ──────────────────────────────────
temp_data = merge_df_off[
    (merge_df_off['operation_id'] == Oper_Code) &
    (merge_df_off['recipe_id']    == Recipe_ID)
].copy()
assert not temp_data.empty, f'recipe_id == {Recipe_ID} 데이터가 없습니다.'

offset_columns = [col for col in temp_data.columns if 'OFFSET' in col]
temp_data.fillna(value={col: 0 for col in offset_columns}, inplace=True)

temp_data['eq_recipe'] = temp_data['eqp_id'] + '//' + temp_data['recipe_id']
temp_data.drop_duplicates(inplace=True)

temp_data['Pol_Time'] = temp_data[Pol_Para].sum(axis=1)
temp_data.dropna(subset=[Thk_Para], inplace=True)

# 계수 결측 보간: 같은 장비의 평균 → 전체 평균 (compute_offset 과 동일)
b_cols_all = [col for col in temp_data.columns if '_B0' in col or '_B1' in col]
temp_data.fillna(temp_data.groupby(['eq_recipe'])[b_cols_all].transform('mean'), inplace=True)
temp_data[b_cols_all] = temp_data[b_cols_all].apply(pd.to_numeric, errors='coerce').fillna(temp_data[b_cols_all].mean())

# ── RR 실측 / RR_Pad 예측 ────────────────────────────────────────────
if 'REV' in Thk_Para:
    temp_data['RR'] = (temp_data[Thk_Para] - Pre_Target) / temp_data['Pol_Time']
else:
    temp_data['RR'] = (Pre_Target - temp_data[Thk_Para]) / temp_data['Pol_Time']

temp_data['RR_Pad'] = temp_data[Pad_Para] * temp_data[APC_Para + '_B1'] + temp_data[APC_Para + '_B0']

# ── OFFSET 계산 ──────────────────────────────────────────────────────
delta = (Target - Pre_Target) if 'REV' in Thk_Para else (Pre_Target - Target)
temp_data['OFFSET_raw'] = delta / temp_data['RR'] - delta / temp_data['RR_Pad']   # clip 전 (교육용 보존)
temp_data['OFFSET']     = temp_data['OFFSET_raw'].clip(-5, 3)
temp_data.loc[temp_data['IDLE'].str.contains('LC_T_|LC_TB_'), 'OFFSET'] = 0

temp_data['recipe_group'] = temp_data['recipe_id'].apply(lambda x: '_'.join(x.split('_')[:3]))
temp_data['APC_Para']     = APC_Para

n_clip = (temp_data['OFFSET_raw'] != temp_data['OFFSET']).sum()
print(f'temp_data: {len(temp_data):,}행')
print(f'RR 실측  평균 {temp_data["RR"].mean():.2f} Å/sec | RR_Pad 예측 평균 {temp_data["RR_Pad"].mean():.2f} Å/sec')
print(f'OFFSET   평균 {temp_data["OFFSET"].mean():+.3f}초 | 표준편차 {temp_data["OFFSET"].std():.3f}초')
print(f'clip(-5, 3) 적용: {n_clip}건 ({n_clip / len(temp_data) * 100:.2f}%)')
temp_data[['Date', 'eqp_id', 'IDLE', 'RR', 'RR_Pad', 'OFFSET_raw', 'OFFSET']].head(5)

In [ ]:
# ── Step 2 시각화 — 실측 RR vs 모델 예측 RR, OFFSET 분포 ─────────────
DEMO_KEY = temp_data.groupby('eq_recipe').size().idxmax()
DEMO_EQP = DEMO_KEY.split('//')[0]
demo = temp_data[temp_data['eq_recipe'] == DEMO_KEY].sort_values('Date')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Step 2 — OFFSET = delta/RR − delta/RR_Pad',
             fontsize=12, fontweight='bold', color=INK_PRIMARY, x=0.01, ha='left')

# ① 대표 장비: 실측 RR vs 모델 예측 RR_Pad
ax = axes[0]
ax.scatter(demo['Date'], demo['RR'], s=5, alpha=0.35, color=RAW_COLOR,
           linewidths=0, label='RR 실측')
ax.plot(demo['Date'], demo['RR_Pad'], color=PALETTE[0], linewidth=1.6, label='RR_Pad 모델 예측')
idle_pts = demo[demo['IDLE'].str.contains('Idle_|Layer_')]
if not idle_pts.empty:
    ax.scatter(idle_pts['Date'], idle_pts['RR'], s=22, alpha=0.9, color=PALETTE[5],
               linewidths=0, label='Idle/Layer 직후 웨이퍼', zorder=5)
style_ax(ax, title=f'{DEMO_EQP} — 실측 vs 모델 예측 RR', ylabel='RR (Å/sec)', xlabel='시간')
ax.tick_params(axis='x', rotation=15)

# ② OFFSET 분포 (clip 경계 표시)
ax = axes[1]
ax.hist(temp_data['OFFSET_raw'], bins=80, color=RAW_COLOR, alpha=0.8, label='clip 전')
ax.hist(temp_data['OFFSET'],     bins=80, color=PALETTE[0], alpha=0.55, label='clip 후')
ax.axvline(-5, color=PALETTE[5], linewidth=1.3, linestyle='--')
ax.axvline(3,  color=PALETTE[5], linewidth=1.3, linestyle='--', label='clip 경계 (−5, +3)')
ax.axvline(0,  color=BASELINE, linewidth=1.2)
style_ax(ax, title='OFFSET 분포', xlabel='OFFSET (초)', ylabel='건수')

plt.tight_layout()
plt.show()
print('→ Idle/Layer 직후 웨이퍼의 실측 RR 이 모델 예측선에서 벗어난 정도가 OFFSET 의 원천입니다.')

## Step 3 — Idle_/Layer_ 구간 집계 → 학습값 저장

전체 웨이퍼 중 **`Idle_` / `Layer_` 라벨이 붙은 웨이퍼만** 골라,
`장비 × 레시피 × IDLE 유형` 그룹별로 OFFSET 을 평균합니다.

```python
# compute_offset() 내부
temp_data_idle = temp_data[IDLE 가 'Idle_' 또는 'Layer_' 포함]

grouped        = temp_data_idle.groupby(['eqp_id', 'recipe_id', 'IDLE']).size()
filtered       = grouped[grouped >= 5]                  # 5건 미만 그룹은 신뢰 불가 → 제외
idle_table     = 그룹별 OFFSET 평균                      # → MongoDB 저장
```

**5건 기준**: Idle 이벤트는 드물어서 그룹당 표본이 적습니다. 1~2건 평균은 개별 웨이퍼
노이즈에 좌우되므로 최소 5건 이상 모인 그룹만 학습값으로 인정합니다.

In [ ]:
# ── Idle_/Layer_ 필터 & 그룹 집계 (compute_offset 과 동일) ───────────
temp_data_idle = temp_data[
    temp_data['IDLE'].str.contains('Idle_') | temp_data['IDLE'].str.contains('Layer_')
]
print(f'Idle_/Layer_ 웨이퍼: {len(temp_data_idle):,}건 / 전체 {len(temp_data):,}건')

grouped         = temp_data_idle.groupby(['eqp_id', 'recipe_id', 'IDLE']).size().reset_index(name='count')
filtered_groups = grouped[grouped['count'] >= 5]
print(f'그룹 {len(grouped)}개 중 5건 이상: {len(filtered_groups)}개 (미달 그룹은 학습값 제외)')
display(grouped.sort_values('count', ascending=False).head(10))

filtered_data = temp_data_idle.merge(filtered_groups[['eqp_id', 'recipe_id', 'IDLE']],
                                     on=['eqp_id', 'recipe_id', 'IDLE'])
idle_table    = filtered_data.groupby(['eqp_id', 'recipe_id', 'IDLE'])['OFFSET'].mean().reset_index()
idle_table['IDLE'] = idle_table['IDLE'].replace('', 'Normal')

if idle_table.empty:
    print('\n→ 5건 이상 그룹이 없어 운영 기준으로는 Idle/Layer 학습값이 저장되지 않습니다.')
    print('  (아래 시각화는 기준 미달 그룹까지 교육용으로 참고 표시합니다)')
else:
    print(f'\nIdle/Layer OFFSET 학습값: {len(idle_table)}건')
    display(idle_table)

In [ ]:
# ── Step 3 시각화 — IDLE 유형별 OFFSET (기준 미달 시 교육용 참고 표시) ──
plot_src   = filtered_data if not idle_table.empty else temp_data_idle
title_note = '' if not idle_table.empty else '  ※ 5건 미달 — 교육용 참고 (운영은 미저장)'

if not plot_src.empty:
    pv = plot_src.pivot_table(index='IDLE', columns='eqp_id', values='OFFSET', aggfunc='mean')
    pv = pv.reindex(sorted(pv.index))

    fig, ax = plt.subplots(figsize=(max(9, 1.4 * len(pv)), 4.5))
    n_eqp = len(pv.columns)
    width = 0.8 / max(n_eqp, 1)
    xpos = np.arange(len(pv))
    for j, eqp in enumerate(pv.columns):
        ax.bar(xpos + (j - (n_eqp - 1) / 2) * width, pv[eqp], width,
               label=eqp, color=PALETTE[j % len(PALETTE)], alpha=0.85)
    # 개별 웨이퍼 산점 겹쳐 그리기 (평균이 어떤 분포에서 나왔는지)
    for i, idle in enumerate(pv.index):
        pts = plot_src[plot_src['IDLE'] == idle]['OFFSET']
        ax.scatter(np.full(len(pts), i), pts, s=12, color=INK_MUTED, alpha=0.55, zorder=5, linewidths=0)
    ax.axhline(0, color=BASELINE, linewidth=1.2)
    ax.set_xticks(xpos)
    ax.set_xticklabels(pv.index, rotation=15)
    style_ax(ax, title=f'Step 3 — IDLE 유형별 OFFSET (막대=평균, 점=개별 웨이퍼){title_note}',
             ylabel='OFFSET (초)')
    plt.tight_layout()
    plt.show()
    print('→ 양수 = 그 Idle 상황에서 연마 시간을 더 줘야 함, 음수 = 줄여야 함')
else:
    print('Idle_/Layer_ 데이터가 없습니다 — 이 공정/기간에는 Idle 이벤트가 없었습니다.')

## Step 4 — `compute_lc_offset()` : LC 계열(패드 교체 등) 별도 집계

패드 교체(`LC_`) 직후는 일반 Idle 보다 RR 변화가 크고 발생 빈도는 더 낮아,
별도 로직으로 집계합니다.

```python
# compute_lc_offset() 내부
temp_data_lc = IDLE 가 'LC_' 포함 (단, _ADD_/_T_/_TB_ 제외)     # 일반 패드 교체
temp_data_tb = IDLE 가 '_ADD_'/'_T_'/'_TB_' 포함                # Truing/추가 작업 계열

if Offset_Group == 'Y':          # Set-up 옵션: 레시피 그룹 단위 통합
    IDLE 라벨에서 장비 번호 부분 제거 → 같은 유형끼리 묶임

lc_offset = _build_idle_table(combined)
#   ① eq_recipe_apc × IDLE 별 OFFSET 평균 (OFFSET_Origin)
#   ② 전체 (장비 × IDLE) 조합 생성 — 데이터 없는 조합도 행으로 만듦
#   ③ recipe_group × IDLE 평균으로 최종 OFFSET 결정
#      → 특정 장비에 LC 이력이 없어도 같은 레시피 그룹의 값을 대신 사용
```

**②③이 핵심**: LC 는 드문 이벤트라 장비별 표본이 부족합니다. 그래서 장비 개별
평균(`OFFSET_Origin`) 대신 **레시피 그룹 평균**을 최종값으로 쓰고, 이력이 전혀 없는
장비에도 그룹 평균을 부여합니다.

In [ ]:
# ── LC / TB 분리 (compute_lc_offset 과 동일) ─────────────────────────
temp_full = temp_data.copy()
temp_full['eq_recipe_apc'] = temp_full['eq_recipe'] + '//' + temp_full['APC_Para']
temp_full['IDLE'] = temp_full['IDLE'].fillna('Normal')

temp_data_lc = temp_full[
    temp_full['IDLE'].str.contains('LC_') &
    ~temp_full['IDLE'].str.contains('_ADD_') &
    ~temp_full['IDLE'].str.contains('_T_') &
    ~temp_full['IDLE'].str.contains('_TB_')
].copy()
temp_data_tb = temp_full[
    temp_full['IDLE'].str.contains('_ADD_') |
    temp_full['IDLE'].str.contains('_T_') |
    temp_full['IDLE'].str.contains('_TB_')
].copy()
print(f'LC 계열(패드 교체): {len(temp_data_lc):,}건 | Truing/추가 계열: {len(temp_data_tb):,}건')
if not temp_data_lc.empty:
    print(f'LC 라벨 예시: {sorted(temp_data_lc["IDLE"].unique())[:5]}')

# ── Offset_Group='Y' 이면 IDLE 라벨에서 장비 번호 제거 후 그룹화 ─────
if Offset_Group == 'Y':
    temp_data_lc['IDLE'] = temp_data_lc['IDLE'].apply(lambda x: '_'.join([x.split('_')[0], x.split('_')[2]]))
    temp_data_tb['IDLE'] = temp_data_tb['IDLE'].apply(lambda x: '_'.join([x.split('_')[0], x.split('_')[3]]))
    print(f'Offset_Group=Y → 그룹화된 LC 라벨: {sorted(temp_data_lc["IDLE"].unique())[:5]}')

combined = pd.concat([temp_data_lc, temp_data_tb], axis=0)

# ── _build_idle_table() 실제 함수 호출 ───────────────────────────────
lc_offset = OFFSET_Get._build_idle_table(combined)
lc_offset['OFFSET'] = lc_offset['OFFSET'].fillna(0)

print(f'\nLC OFFSET 학습값: {len(lc_offset)}건 (전체 장비 × IDLE 조합)')
display(lc_offset[['eqp_id', 'recipe_id', 'IDLE', 'OFFSET_Origin', 'OFFSET']]
        .sort_values(['IDLE', 'eqp_id']).reset_index(drop=True))

In [ ]:
# ── Step 4 시각화 — 장비 개별 평균 vs recipe_group 평균 ──────────────
if not lc_offset.empty:
    plot_lc = lc_offset.sort_values(['IDLE', 'eqp_id']).reset_index(drop=True)
    labels  = plot_lc['eqp_id'] + '\n' + plot_lc['IDLE']
    xpos    = np.arange(len(plot_lc))

    fig, ax = plt.subplots(figsize=(max(10, 0.9 * len(plot_lc)), 4.5))
    ax.bar(xpos - 0.18, plot_lc['OFFSET_Origin'].fillna(0), 0.36,
           label='장비 개별 평균 (OFFSET_Origin)', color=RAW_COLOR)
    ax.bar(xpos + 0.18, plot_lc['OFFSET'], 0.36,
           label='최종 학습값 (recipe_group 평균)', color=PALETTE[0])
    # 데이터가 아예 없어 그룹 평균으로 채워진 조합 표시
    for i, row in plot_lc.iterrows():
        if pd.isna(row['OFFSET_Origin']):
            ax.text(i, 0, '이력\n없음', ha='center', va='bottom', fontsize=7, color=PALETTE[5])
    ax.axhline(0, color=BASELINE, linewidth=1.2)
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels, fontsize=7.5)
    style_ax(ax, title='Step 4 — LC OFFSET: 장비 개별 평균 → recipe_group 평균으로 보강',
             ylabel='OFFSET (초)')
    plt.tight_layout()
    plt.show()
else:
    print('LC 계열 데이터가 없습니다 — 이 공정/기간에는 패드 교체 이벤트가 없었습니다.')

## 최종 — 운영 함수 전체 실행 & 학습값 확인

실제 운영은 `Module_Get.compute_offset()` 이 모든 학습 단위(FB_Type=TIME 행)에 대해
`load_rr_data → compute_offset → compute_lc_offset` 을 실행하고, 결과를 MongoDB 컬렉션
**`MICO_OFFSET_{Lot_Code}_{Oper_Desc}_{Fab}`** 에 저장합니다.

APC 는 연마 시간을 계산할 때 이 학습값을 다음처럼 사용합니다:

```
연마 시간 = 목표 제거량 / RR_Pad(RR 학습값)  +  OFFSET(현재 Idle 유형의 학습값)
```

In [ ]:
# ── 이전 OFFSET 결과 초기화 후 운영 함수 전체 실행 ───────────────────
off_collection = 'MICO_OFFSET_' + Lot_Code + '_' + Oper_Desc + '_' + Fab
_STORE[off_collection] = []

Module_Get.compute_offset(merge_df.copy(), mico_info_key, pol_type)

off_result = pd.DataFrame(_STORE.get(off_collection, []))
assert not off_result.empty, 'OFFSET 학습값이 생성되지 않았습니다. 데이터/Set-up 조건을 확인하세요.'

print(f'\n최종 OFFSET 학습값: {len(off_result)}건 (컬렉션: {off_collection})')
show_cols = [c for c in ['APC_Para', 'eqp_id', 'recipe_id', 'IDLE', 'OFFSET'] if c in off_result.columns]
display(off_result[show_cols].sort_values(['IDLE', 'eqp_id']).reset_index(drop=True))

In [ ]:
# ── 최종 시각화 — IDLE 유형별 학습값 요약 ────────────────────────────
summary = (off_result.groupby('IDLE')['OFFSET']
           .agg(['mean', 'std', 'count'])
           .sort_values('mean'))

fig, ax = plt.subplots(figsize=(max(9, 1.2 * len(summary)), 4.8))
bars = ax.bar(summary.index, summary['mean'], color=PALETTE[0], width=0.6,
              yerr=summary['std'].fillna(0), ecolor=INK_MUTED, capsize=3)
ax.bar_label(bars, fmt='%+.2f', fontsize=8, padding=4, color=INK_SECONDARY)
ax.axhline(0, color=BASELINE, linewidth=1.2)
style_ax(ax, title=f'최종 OFFSET 학습값 — {FOR_KEY} (IDLE 유형별 평균 ± 표준편차)',
         ylabel='OFFSET (초)', legend=False)
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()
print('양수 = 그 상황에서 연마 시간 가산, 음수 = 감산.')
for idle, row in summary.iterrows():
    print(f'  {idle:<14}: {row["mean"]:+.3f}초 (장비×레시피 {int(row["count"])}건)')

## 정리

| Step | 함수 | 입력 → 출력 |
|---|---|---|
| 준비 | `compute_pre_thk_vm()` → `compute_removal_rate()` | 교육 자료 1·2편 — RR 학습값(B1/B0) 준비 |
| 1 | `OFFSET_Get.load_rr_data()` | RR 학습값 → `{APC}_B1/_B0` 컬럼 결합 (weighted 우선, merge_asof) |
| 2 | (`compute_offset()` 내부) | 실측 RR vs RR_Pad → OFFSET (초), clip(−5, 3) |
| 3 | (`compute_offset()` 내부) | Idle_/Layer_ 그룹(≥5건) 평균 → 학습값 저장 |
| 4 | `compute_lc_offset()` + `_build_idle_table()` | LC 계열 → recipe_group 평균으로 보강 저장 |

세 교육 자료의 관계를 정리하면:

```
1편 Pre_Thk_VM : 투입 전 두께를 추정          →  RR 계산의 입력
2편 Removal Rate: 정상 상태의 연마율 모델 학습  →  연마 시간의 기본값
3편 Offset      : Idle 상황별 시간 보정량 학습  →  연마 시간의 보정값

APC 연마 시간 = 목표 제거량 / (B1×소모량 + B0)  +  OFFSET(Idle 유형)
```

> 다른 공정을 보려면 맨 위 [실행 설정] 셀에서 `FAMILY` / `OPER_DESC`
> (필요시 `KEY_INDEX` / `APC_INDEX`)만 바꾸고 전체 재실행하면 됩니다.